In [1]:
# SPDX-License-Identifier: CC-BY-4.0
# Code for "Active Continual Learning with Metaplastic Binary Bayesian Neural Networks"
# Kellian Cottart, Théo Ballet, Djohan Bonnet, Damien Querlioz
# Portions of the code are adapted from the Pytorch project (BSD-3-Clause)
# Author: Kellian Cottart <kellian.cottart@gmail.com>
# Date: 2025-30-01

In [2]:

import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import numpy as np
import os
import seaborn as sns
import re
import json
import pandas as pd
AXESSIZE = 28
FONTSIZE = 26
TICKSIZE = 24   
LEGENDSIZE = 26
plt.rcParams['svg.fonttype'] = 'none'
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.05"
results_folder = "results-appendix-openloris-al-budget"
figures_folder = "fig-appendix-openloris-al-budget"
os.makedirs(figures_folder, exist_ok=True)
df = pd.DataFrame()
# iterate through all root folders in the results folder
for folder in os.listdir(results_folder):
    current_path = os.path.join(results_folder, folder)
    # extract the name from the first config
    config_path = os.path.join(current_path, "config0/config.json")
    with open(config_path, "r") as f:
        config = json.load(f)
    # n_iterations is the number of config folders
    n_iterations = len([f for f in os.listdir(current_path) if f.startswith("config") and os.path.isdir(os.path.join(current_path, f))])
    
    # Add the row of parameters to the dataframe   
    def method_wrapper(config):
        if "active_learning" in config["network_params"]:
            if not "mode" in config["network_params"]["active_learning"]:
                return list(config["network_params"]["active_learning"].keys())[0]  
            else:
                mode_map = {
                    0: "Epistemic",
                    1: "Aleatoric",
                    2: "Predictive",
                    3: "Variation ratio",
                    4: "Random",
                    5: "Var. w/ labels",
                    6: "Budgeted Variation ratio"
                }
                return mode_map.get(config["network_params"]["active_learning"]["mode"], "Random")
        return "Random"
    
    # Add the row of parameters to the dataframe   
    field = [key for key in config.keys() if "ewc" in key]
    row = {
        "path": current_path,
        "opt": config["optimizer"],
        "n_tasks": config["n_tasks"],
        "n_epochs": config["epochs"],
        "n_train_samples": config["n_train_samples"] if "n_train_samples" in config else 1,
        "n_test_samples": config["n_test_samples"] if "n_test_samples" in config else 1,
        "n_iterations": n_iterations,
        "batch_size": config["train_batch_size"],
        # key containing ewc but not strictly equal to ewc
        "ewc": config[field[0]] if len(field) > 0 else "none",
        "method": method_wrapper(config),
        "n_active_learning_samples": int(config["network_params"]["active_learning"]["samples"]) if "active_learning" in config["network_params"] and "samples" in config["network_params"]["active_learning"] else 0,
        "threshold": float(config["network_params"]["active_learning"]["threshold"]) if "active_learning" in config["network_params"] and "threshold" in config["network_params"]["active_learning"] else 0.0,
        "N": int(config["optimizer_params"]["N"]) if "N" in config["optimizer_params"] else "none",
        "interleaved": int(config["task_params"].get("interleaved", 0)),
        "n_splits_per_epoch": int(config.get("n_splits_per_epoch", 1)),
        "budget": float(config["network_params"]["active_learning"]["budget"]*100) if "active_learning" in config["network_params"] and "budget" in config["network_params"]["active_learning"] else 0.0,
        "scaler": float(config["network_params"]["active_learning"]["scaler"]) if "active_learning" in config["network_params"] and "scaler" in config["network_params"]["active_learning"] else 0.0
    }
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

In [3]:
data = []
for idx, row in df.iterrows():
    path = row["path"]
    n_tasks = row["n_tasks"]
    n_epochs = row["n_epochs"]
    n_iterations = row["n_iterations"]
    n_splits_per_epoch = row["n_splits_per_epoch"] if "n_splits_per_epoch" in row else 1
    full_accuracies = []
    full_updates = []
    final_updates = []
    full_wras = []
    full_uras = []
    
    for it in range(n_iterations):
        current_it_path = os.path.join(path, f"config{it}")
        accuracy_path = os.path.join(current_it_path, "accuracy")
        uncertainty_path = os.path.join(current_it_path, "uncertainty")
        accuracies = []
        updates = []
        wras = []
        uras = []
        for task in range(n_tasks):
            for epoch in range(n_epochs):
                for split in range(n_splits_per_epoch):
                    suffix = f"split={split}-task={task}-epoch={epoch}.npy"
                    accuracies.append(jnp.load(os.path.join(accuracy_path, suffix)))
                    updates.append(jnp.load(os.path.join(accuracy_path, "iterations-"+suffix)))
                    wras.append(jnp.load(os.path.join(accuracy_path, "wra-"+suffix)))
                    uras.append(jnp.load(os.path.join(accuracy_path, "ura-"+suffix)))
        full_accuracies.append(np.array(accuracies))
        final_updates.append(jnp.load(os.path.join(current_it_path, "iterations.npy")))
        full_updates.append(np.array(updates))
        full_wras.append(np.array(wras))
        full_uras.append(np.array(uras))

    full_accuracies = np.array(full_accuracies)*100
    full_wras = np.array(full_wras)*100
    full_uras = np.array(full_uras)*100
    full_updates = np.array(full_updates)
    
    final_updates_mean = np.array(final_updates).mean()
    final_updates_std = np.array(final_updates).std()
    accuracy_array_mean = np.mean(full_accuracies, 0)[-1, :].mean()
    accuracy_array_std = np.mean(full_accuracies[:, -1, :], -1).std()
    data.append((accuracy_array_mean, accuracy_array_std, full_updates, full_accuracies, final_updates_mean, final_updates_std, full_wras, full_uras))

# Add new columns to df
df["accuracies_mean"] = [d[0] for d in data]
df["accuracies_std"] = [d[1] for d in data]
df["updates"] = [d[2] for d in data]
df["accuracies"] = [d[3] for d in data]
df["final_updates"] = [d[4] for d in data]
df["final_updates_std"] = [d[5] for d in data]
df["wras"] = [d[6] for d in data]
df["uras"] = [d[7] for d in data]


In [4]:
# make a subdf with only final updates and accuracies
subdf = df[df["method"] == "Budgeted Variation ratio"]
# compute percentage
max_ex = df["final_updates"].max()
# print max accuracy for random
random_df = df[df["method"] == "Random"]
max_random_accuracy = random_df["accuracies_mean"].max()
print(f"Max random accuracy: {max_random_accuracy:.2f}%")

# add random df to subdf
subdf = pd.concat([subdf, random_df], ignore_index=True)

# scale final_updates to percentage of max_ex for variation-ratio rows
subdf["final_updates"] = np.array(subdf["final_updates"]) / max_ex * 100
subdf["final_updates_std"] = np.array(subdf["final_updates_std"]) / max_ex * 100
subdf = subdf[["method", "final_updates", "final_updates_std", "accuracies_mean", "accuracies_std", "n_active_learning_samples", "budget", "scaler"]]
subdf = subdf.sort_values("final_updates")
print(subdf.to_latex(index=False, float_format="%.2f"))



Max random accuracy: 87.76%
\begin{tabular}{lrrrrrrr}
\toprule
                  method &  final\_updates &  final\_updates\_std &  accuracies\_mean &  accuracies\_std &  n\_active\_learning\_samples &  budget &  scaler \\
\midrule
Budgeted Variation ratio &           0.50 &               0.00 &            52.68 &            2.29 &                         10 &    0.50 &    0.01 \\
Budgeted Variation ratio &           0.55 &               0.01 &            67.53 &            2.10 &                         10 &    0.50 &    0.05 \\
Budgeted Variation ratio &           1.00 &               0.00 &            61.95 &            1.95 &                         10 &    1.00 &    0.01 \\
Budgeted Variation ratio &           1.01 &               0.00 &            76.36 &            1.40 &                         10 &    1.00 &    0.05 \\
Budgeted Variation ratio &           1.11 &               0.03 &            79.84 &            0.65 &                         10 &    0.50 &    0.10 \\
Budgeted